# 1 · Your first Markov chain

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sem-16161d/GreasyPidginTutorial/blob/main/EN/jupyter/markov-book-en/01_first_markov_chain.ipynb)

A **Markov chain** builds a sequence one step at a time. At every step it looks only at the last
few symbols (its *memory*) and picks the next symbol with certain probabilities.

In this chapter there is no music yet. We use weather and letters, so you can see how little is
needed, and meet the three GreasyPidgin objects we will use all day:

- `MarkovRule` — one row of the recipe: *"after this, one of those comes next, with these weights"*
- `MarkovRules` — a collection of rules
- `MarkovChain` — follows the rules and produces a sequence

Run the cells from top to bottom with **Shift + Enter**. The first cell installs everything.

In [ ]:
# Setup -- runs by itself (about 20 seconds the first time)
import importlib.util, pathlib, subprocess, sys, urllib.request

GP_REF        = "main"   # which version of GreasyPidgin to use (a git branch or tag)
TUTORIAL_REF  = "main"   # which version of this tutorial's helper file to use
GP_REPO       = "sem-16161d/GreasyPidgin"
TUTORIAL_REPO = "sem-16161d/GreasyPidginTutorial"

if importlib.util.find_spec("GreasyPidgin") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                    f"git+https://github.com/{GP_REPO}.git@{GP_REF}"], check=True)

if not pathlib.Path("workshop_helpers.py").exists():
    urllib.request.urlretrieve(
        f"https://raw.githubusercontent.com/{TUTORIAL_REPO}/{TUTORIAL_REF}/EN/jupyter/markov-book-en/workshop_helpers.py",
        "workshop_helpers.py")

import random
import matplotlib.pyplot as plt
from GreasyPidgin import MarkovChain, MarkovRule, MarkovRules
import workshop_helpers as wh

print("Ready!")

## A chain you write by hand: the weather

Each rule says what can follow a state, and how likely each option is.
The weights are normalised automatically, so `[6, 3, 1]` works just as well as `[0.6, 0.3, 0.1]`.

In [ ]:
rules = MarkovRules([
    MarkovRule("sunny",  ["sunny", "cloudy", "rainy"], [0.6, 0.3, 0.1]),
    MarkovRule("cloudy", ["sunny", "cloudy", "rainy"], [0.3, 0.4, 0.3]),
    MarkovRule("rainy",  ["sunny", "cloudy", "rainy"], [0.1, 0.4, 0.5]),
])
chain = MarkovChain(rules=rules)
print(rules)

Now let the chain run for 14 days. `seed` is *where the chain starts* (not a random seed),
`n` is how many new steps to take. Run the cell several times: every run gives a different week.

In [ ]:
days = chain.getNElements(seed="sunny", n=14)
print(" -> ".join(days))

**Try it:** change the weights so that rain almost never ends. Then run the cell above again
and watch how the weather changes.

### Seeing the rules

A table of probabilities is easier to read as a picture. Each row is the *current* state,
each column the *next* one; the brighter, the more likely.

In [ ]:
wh.plot_transition_matrix(chain, label=str)
plt.show()

## Learning a chain from data

Writing rules by hand gets tiring. A chain can also **learn its rules from an existing sequence**
by counting what followed what. Here the sequence is a word, one letter per step.

In [ ]:
word = list("abracadabra")

learned = MarkovChain()
learned.generateRulesFromList(word, order=1)   # order 1: remember only the last letter
print(learned.rules)

In [ ]:
new_word = learned.getNElements(seed="a", n=25)
print("".join(new_word))

After an `a` the original word continues with `b`, `c`, `d` or (at the very end) wraps around
to `a`. The chain only knows that — so it invents new words with the same "taste".

### Memory: order 2

With `order=2` the chain remembers the last **two** letters. The seed must then have two letters, too.

In [ ]:
learned2 = MarkovChain()
learned2.generateRulesFromList(word, order=2)

print("".join(learned2.getNElements(seed=("a", "b"), n=25)))

**Try it:**

- Replace `"abracadabra"` with a sentence of your own, e.g. `list("the quick brown fox")`.
- Compare `order=1`, `2`, `3`. What happens to the output when the order gets large?
  (Hint: with a lot of memory and little data, the chain has only one choice at every step —
  it can only copy the original.)

That trade-off between *copying* and *inventing* is the main idea of this workshop.
In the next chapter we do the same with notes from a MIDI file.